# Demo 01 — From prompt to product: the support assistant

**One disease, four cures.** We start with a bare LLM that confidently lies, then add the four
layers that turn it into an application — **structured output → tools → retrieval → the loop** —
all on the *same* example, with every step visible in an MLflow trace.

> Run order: top to bottom. Set `USE_TEST_MODEL=1` in `.env` to rehearse without a live model.
> The "hallucination" beat only really lands on a **real** model.

## Setup

In [ ]:
# --- Setup: env, MLflow tracing, and a model factory -------------------------
import os
import nest_asyncio

from dotenv import load_dotenv

import mlflow
import mlflow.pydantic_ai

from pydantic_ai import Agent, NativeOutput
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.models.test import TestModel

load_dotenv()                        # reads ../.env
nest_asyncio.apply()                 # lets `await` work inside Jupyter cells

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT", "session7-app-dev"))
mlflow.pydantic_ai.autolog()         # type: ignore # one line: every agent/LLM/tool/MCP call -> a trace span
mlflow.tracing.disable_notebook_display() # type: ignore

USE_TEST_MODEL = os.getenv("USE_TEST_MODEL", "0") == "1"

def make_model():
    """Real model from your env (vLLM/Ollama, OpenAI-compatible), or a fake one for rehearsal.

    The live model MUST support tool/function calling AND structured output — a small
    model (e.g. llama3.2:1b) will fail Layers 1-4. Use qwen2.5:7b / llama3.1:8b or larger.
    """
    if USE_TEST_MODEL:
        return TestModel()            # canned responses, no live LLM needed
    return OpenAIChatModel(           # OpenAIModel was renamed -> OpenAIChatModel
        os.environ.get("MODEL", "qwen2.5:7b"),
        provider=OpenAIProvider(
            base_url=os.environ.get("OPENAI_BASE_URL", "http://localhost:11434/v1"),
            api_key=os.environ.get("OPENAI_API_KEY", "not-needed-for-local"),
        ),
    )

def show_trace():
    """Flush + print the span tree of the last run (full timeline lives in the MLflow UI)."""
    mlflow.flush_trace_async_logging()
    tr = mlflow.get_trace(mlflow.get_last_active_trace_id()) # type: ignore
    for s in tr.data.spans:
        print(f"  [{s.span_type:5}] {s.name}")
    print("tokens:", getattr(tr.info, "token_usage", "n/a"))
    print("Open the full timeline:", mlflow.get_tracking_uri())

print("setup ready | USE_TEST_MODEL =", USE_TEST_MODEL)


/private/tmp/claude-502/-Users-ponkgir-code-AI-LLM-Fundamentals-Workshop-Series/6c5d059c-f7ed-4484-987b-2a0e717669fe/scratchpad/s7venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup ready | USE_TEST_MODEL = False


## The disease — a bare LLM

No tools, no data, free-form text out. Ask it about a specific order. On a real model it answers
**warmly, specifically, and entirely from imagination** — it never looked anything up, and it hands
your UI a paragraph your code can't act on. Four failures in one breath.

In [2]:
bare = Agent(make_model(), instructions="You are a helpful order-support assistant.")
r = bare.run_sync("Where's my order #A7731, and can I return part of it?")
print(r.output)

To help you with your query about order #A7731 and returns, I'll need some more information:

1. **Customer Information**: Could you please provide the email or phone number associated with this order?
2. **Order Details**: When was this order placed? What items are included in the order?

Regarding returning part of your order, here’s a general guidance:
- Many retailers have specific return policies that govern partial returns.
- Typically, you can return products if they are defective or not as described.
- For returns, details like return deadlines and shipping costs need to be considered.

Once I have more information, I can provide more accurate assistance.


## Layer 1 — Structured output (the contract problem)

Software talks through fixed contracts (REST bodies, function signatures, DB schemas). You can't
feed free prose into the next block. So make the model **fill out a form, not write a letter**:
constrain its output to a Pydantic schema. (This is Session 3's constrained decoding, in service of
an application contract.) Note: this guarantees the *shape*, not the *truth* — the `order_id` can
still be invented. That's what Layers 2–3 fix.

We use structured output for the **extraction/classification** step. The conversational assistant in
the next layers deliberately replies in **prose** — you constrain the data you parse, not the final
human-facing message (over-constraining that is the one real failure mode of structured output).

In [3]:
from pydantic import BaseModel

class SupportReply(BaseModel):
    intent: str                 # e.g. "order_status", "return_request", "both"
    order_id: str | None
    needs_clarification: bool
    message: str

# NativeOutput = the provider constrains decoding to the schema (Session 3's idea, live).
# Default "tool" output-mode is unreliable on local models served via Ollama; NativeOutput
# (JSON-schema response_format) is the robust path. PromptedOutput is the portable fallback.
structured = Agent(make_model(), output_type=NativeOutput(SupportReply),
                   instructions="Classify the request and extract fields. Ask to clarify if ambiguous.")
r = structured.run_sync("Where's my order #A7731, and can I return part of it?")
print(type(r.output).__name__, "->", r.output)


SupportReply -> intent='status_change' order_id='A7731' needs_clarification=False message='Can I return part of the order?'


## Layer 2 — Tool / function calling

The model's knowledge is frozen and it can't see your systems. So let it **reach out**: describe a
function, and instead of prose it emits a structured *call*; **your code** runs it and hands back the
real result. The model is the **decision layer** (which function, what args); your code is the
**execution layer**. It never touches your DB — that's the whole security story in one line.

In [4]:
# Mock "database" — stands in for your real systems
ORDERS = {
    "A7731": {"status": "delivered", "delivered_on": "2026-06-22",
              "items": [{"sku": "AZ-4471", "name": "Wireless Earbuds", "opened": True}]},
}

# The tool-using assistant replies in PROSE (no output_type). We constrain the *extraction*
# step (Layer 1 above), not the final user-facing reply — forcing a schema here would make
# the model emit the schema instead of calling the tool (over-constraining; see the note above).
agent = Agent(make_model(),
              instructions="Use tools to look up real facts before answering. Never guess order facts.")

@agent.tool_plain
def get_order(order_id: str) -> dict:
    """Look up an order by its ID; returns status, delivery date, and line items."""
    return ORDERS.get(order_id, {"error": "order not found"})

r = agent.run_sync("Where's my order #A7731?")
print(r.output)
print("\nThe model REQUESTED the tool; our code RAN it. Message sequence:")
for m in r.all_messages():
    for part in getattr(m, "parts", []):
        name = getattr(part, "tool_name", "")
        print(f"   {type(part).__name__:16} {name}")


Your order #A7731 was delivered on June 22, 2026. The item in your order was a pair of Wireless Earbuds, which has been opened.

The model REQUESTED the tool; our code RAN it. Message sequence:
   UserPromptPart   
   ToolCallPart     get_order
   ToolReturnPart   get_order
   TextPart         


## Layer 3 — Retrieval (RAG)

"Can I return an *opened* item?" — that fact lives in your **policy**, which the model has never
seen. Retrieve the relevant clause and put it in context: now the answer is grounded in *your* data,
with a citation. (Here retrieval is a one-liner; **demo_03** digs into *why* retrieval is genuinely
hard — a question and its answer don't look alike.)

In [5]:
POLICY = {
    "returns": ("Sealed items: full refund within 30 days. "
                "Unsealed/opened items: exchange only, within 14 days of delivery, no cash refund."),
}

@agent.tool_plain
def get_policy(topic: str) -> str:
    """Retrieve the returns/exchange policy clause for a topic (e.g. 'returns')."""
    return POLICY.get(topic, "No policy found for that topic.")

r = agent.run_sync("My earbuds (order A7731) are opened — can I return them?")
print(r.output)

According to our policy, for opened items like your earbuds (order A7731), you can only request an exchange and not a cash refund. You have 14 days from the delivery date to make this exchange.

Would you like assistance with making an exchange order? Or do you need more information regarding the return/exchange process?


## Layer 4 — The loop

A real request needs *all* of it, in sequence, with judgement: look up the order → check policy →
notice the customer didn't say *which* items → ask → resolve. That control flow **is** the point:

> **An agent is a while-loop around an LLM with tools, retrieval, and a stopping condition.**

`pydantic-ai` runs that loop for you inside `agent.run_sync(...)`. `all_messages()` lets you read the
**decoded execution sequence** — the model's runtime plan — step by step.

In [6]:
r = agent.run_sync("Where's order A7731, and can I return part of it?")
print("FINAL:", r.output, "\n")
print("THE LOOP (runtime plan):")
for m in r.all_messages():
    for part in getattr(m, "parts", []):
        kind = type(part).__name__
        name = getattr(part, "tool_name", "")
        print(f"  {kind:18} {name}")

FINAL: Based on our return policy for sealed and opened items:

- **Sealed (unclosed) Wireless Earbuds**: You can request a full refund within 30 days from the delivery date.
- **Opened Wireless Earbuds**: According to your order details, since these were already opened, you are only eligible for an exchange within 14 days of delivery. Cash refunds are not available for opened items.

Given that your order was delivered on June 22, 2026, and assuming today's date is within the 14-day window after delivery (up to July 5, 2026), you can pursue an exchange but not a cash refund. 

Would you like to proceed with an exchange? If yes, please specify the model or item details you wish to return/exchange. 

THE LOOP (runtime plan):
  UserPromptPart     
  TextPart           
  ToolCallPart       get_order
  ToolReturnPart     get_order
  TextPart           
  ToolCallPart       get_policy
  ToolReturnPart     get_policy
  TextPart           


### The loop made visible — failure & retry

A demo calls the tool once and it works. **Production** handles the tool *throwing*. With `retries`,
the loop feeds the error back to the model so it can adapt instead of crashing. That difference is
the whole game.

In [7]:
flaky = Agent(make_model(), retries=3,
              instructions="Look up the order with the tool. If a tool errors, retry it; then answer.")

from pydantic_ai import ModelRetry
_attempts = {"n": 0}

@flaky.tool_plain
def get_order_status(order_id: str) -> dict:
    """Look up an order's status (first attempt simulates a transient timeout)."""
    _attempts["n"] += 1
    if _attempts["n"] == 1:
        # ModelRetry feeds the error back to the model so the loop can recover
        raise ModelRetry("OrderService timeout (simulated) - please retry")
    return {"order_id": order_id, "status": "delivered", "delivered_on": "2026-06-22"}

r = flaky.run_sync("What's the status of order A7731?")
print(r.output)

# Report what the loop ACTUALLY did (don't assert success blindly).
n = _attempts["n"]
if n >= 2:
    print(f"\nTool attempted {n}x: 1st call errored -> the loop fed the error back -> retried -> recovered.")
else:
    print(f"\nTool attempted {n}x: the model did not retry the tool. "
          f"(Capable models recover here; weaker ones give up — that's the demo-vs-production point.)")


The status of order A7731 is delivered, and it was delivered on June 22, 2026.

Tool attempted 2x: 1st call errored -> the loop fed the error back -> retried -> recovered.


## Observability — the loop as a timeline

`mlflow.pydantic_ai.autolog()` (one line, top of notebook) captured every run as a **trace**. Below
is the span tree inline; the real payoff is the **timeline view in the MLflow UI** — every LLM call,
tool call, latency, and token count, nested. (Callback to Session 4: this is your monitoring story.)

In [8]:
show_trace()

  [AGENT] Agent.run_sync
  [AGENT] Agent.run
  [LLM  ] InstrumentedModel.request
  [TOOL ] ToolManager.handle_call
tokens: {'input_tokens': 700, 'output_tokens': 99, 'total_tokens': 799}
Open the full timeline: http://127.0.0.1:5001


## Recap

Invented status → **tools**. Invented policy → **RAG**. Unparseable paragraph → **structured output**.
One-shot guess → **the loop**. Same failure, four fixes.

**Next (demo_02):** we hand-wired two tools. What if there were fifty, across three apps and three
models? That's the N×M problem — and **MCP** is the standard that fixes it.